# Part 2.8 Middleware

After this notebook you can add human approval, PII redaction, and a call limit to an
agent without touching its prompt.

In [ ]:
from kea.config import MODEL, setup
from kea.notebook import ask_and_approve, final, steps

setup("kea-rebooking-local")

## See it

Middleware hooks into the agent loop. Human approval is one hook: the run stops at
`issue_credit` until a person decides.

<img src="../../assets/diagrams/middleware-hooks.png" alt="Middleware hooks around the agent loop: before_agent, before_model, wrap_model_call, after_model, wrap_tool_call, after_agent" width="520">

<img src="../../assets/screenshots/p2-08-hitl-approval.png" alt="Studio's interrupt card for issuecredit: the proposed arguments and the Approve, Edit, and Reject buttons." width="900">

## Build it

Middleware wraps the agent loop. Three things this agent needs, none of which belong in a
prompt:

- **Human approval** before `issue_credit` runs.
- **PII redaction**, so a card number or an email never reaches the model.
- **A call limit**, so a confused agent can't loop forever.

The prompt states the credit amount, so this agent needs no policy skill.

In [ ]:
from deepagents import create_deep_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, PIIMiddleware
from langgraph.checkpoint.memory import InMemorySaver

from kea.rebooking.tools import TOOLS

guarded = create_deep_agent(
    model=MODEL,
    tools=TOOLS,
    system_prompt=(
        "You are the Kea Air rebooking assistant. A mechanical cancellation delayed 2 to "
        "6 hours earns a $100 credit. Do not ask the passenger to confirm credits."
    ),
    checkpointer=InMemorySaver(),
    interrupt_on={"issue_credit": True},
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_output=True, apply_to_tool_results=True),
        PIIMiddleware(
            "credit_card",
            strategy="redact",
            apply_to_output=True,
            apply_to_tool_results=True,
        ),
        ModelCallLimitMiddleware(run_limit=40, exit_behavior="end"),
    ],
)

Send a message that earns a credit. The run stops at `issue_credit` and hands back the
proposal. Nothing has been issued yet. A human decides. `approve` runs the call as
proposed. `edit` replaces the arguments, for example
`{"type": "edit", "edited_action": {"name": "issue_credit", "args": {...}}}`.
`reject` skips the call and tells the model why. We approve:

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.types import Command

config = {"configurable": {"thread_id": "alex-guarded"}}
message = HumanMessage(
    "KA 1182 was cancelled. Confirmation ABC123, last name Example. Rebook me and give "
    "me what I'm owed."
)
result = await guarded.ainvoke({"messages": [message]}, config)
for request in result["__interrupt__"][0].value["action_requests"]:
    print(request["name"], request["args"])

result = await guarded.ainvoke(Command(resume={"decisions": [{"type": "approve"}]}), config)
final(result)

Now a passenger who pastes a card number and an email. Look at the first message **as the
agent stores it**:

In [ ]:
result = await ask_and_approve(
    guarded,
    "KA 1182 got cancelled, ABC123, last name Example. Refund my card 4532015112830366 "
    "or email me at alex.example@example.com.",
)
print(result["messages"][0].content)

And the call limit. With `run_limit=2` the agent gets two model calls, then the middleware
ends the run with a message instead of letting it loop:

In [ ]:
from kea.rebooking.tools import get_booking, search_flights

limited = create_deep_agent(
    model=MODEL,
    tools=[get_booking, search_flights],
    system_prompt="You are Kea Air's rebooking assistant.",
    middleware=[ModelCallLimitMiddleware(run_limit=2, exit_behavior="end")],
)
result = await ask_and_approve(
    limited,
    "KA 1182 got cancelled. ABC123, last name Example. Look up my booking, then list "
    "every flight on my route today and tomorrow.",
)
steps(result)
print()
final(result)

## Under the hood

<img src="../../assets/slides/middleware.jpg" width="720">

A middleware is a class with hooks into the loop:

| Hook | Runs |
|---|---|
| `before_agent`, `after_agent` | once per run |
| `before_model`, `after_model` | around each model call's inputs and outputs |
| `wrap_model_call` | around each model call, so it can change the request or retry |
| `wrap_tool_call` | around each tool call, so it can block, change, or replace it |

`PIIMiddleware` rewrites messages before the model sees them. Human approval is a hook
around the tool call. `redact` replaces the value, `mask` keeps the last four digits,
`hash` keeps identity without the value, and `block` raises. A rule that is specific to
your business is a few lines. This one refuses any credit above the policy maximum,
whatever the model asks for:

In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage

from kea.rebooking.tools import issue_credit


@wrap_tool_call
async def credit_cap(request, handler):
    call = request.tool_call
    if call["name"] == "issue_credit" and call["args"]["amount_usd"] > 300:
        return ToolMessage("Refused: credits are capped at $300.", tool_call_id=call["id"])
    return await handler(request)


capped = create_deep_agent(
    model=MODEL,
    tools=[issue_credit],
    system_prompt=(
        "You issue credits for supervisors. Do not ask questions: call issue_credit "
        "right away with exactly the amount requested."
    ),
    middleware=[credit_cap],
)
result = await ask_and_approve(capped, "Issue a $500 credit to ABC123 for a delayed flight.")
steps(result)
final(result)

- The hook is `async` because the agent runs with `ainvoke`. A sync-only wrapper fails
  there.
- Returning a `ToolMessage` instead of raising gives the model something it can read and
  recover from.
- Order matters: each middleware sees what the ones before it did.

Docs: [Human-in-the-loop](https://docs.langchain.com/oss/python/deepagents/human-in-the-loop),
[Built-in middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in),
[Custom middleware](https://docs.langchain.com/oss/python/langchain/middleware/custom).

## Recap

- Middleware wraps the agent loop. Approval, privacy, and limits live there, not in the
  prompt.
- `interrupt_on` pauses before a tool and lets a human approve, edit, or reject.
- A business rule is a short `wrap_tool_call` function.

**Next:** [Part 2.9 Assertion evals](09_assertion_evals.ipynb)